# Qa visualization

Purpose: inspect the named geometry, dataset, or model diagnostic.

Prerequisites: install the project with the notebook extra and supply the research artifacts selected in the configuration cells. Launch Jupyter from the project root. See `docs/notebooks.md` for per-notebook inputs.

Outputs: displayed diagnostics and, where configured, exported figures/tables. Run cells from top to bottom. Saved outputs have been cleared.


# QA Visualization: Project Bathymetry + NORA Points

This notebook verifies that:
- the processed site bathymetry field in `data/processed/bathy/bathy_field_project_site.npz` is correctly placed in projected coordinates,
- project-site points from `configs/sites.yaml` align with the local bathymetry structure,
- offshore NORA points are visible relative to the project site bathymetry.

In [ ]:
from pathlib import Path

import contextily as ctx
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from pyproj import CRS, Transformer
from scipy.spatial import cKDTree

plt.rcParams["figure.dpi"] = 120

In [ ]:
def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "configs").exists() and (cwd / "data").exists():
        return cwd
    if (cwd.parent / "configs").exists() and (cwd.parent / "data").exists():
        return cwd.parent
    raise FileNotFoundError("Could not resolve project root containing configs/ and data/.")


def parse_epsg(value) -> int | None:
    if value is None:
        return None
    if isinstance(value, (int, np.integer)):
        return int(value)
    text = str(value).strip()
    if text.lower().startswith("epsg:"):
        text = text.split(":")[-1]
    return int(text) if text.isdigit() else None


def parse_npz_metadata(metadata_obj) -> dict:
    if isinstance(metadata_obj, np.ndarray):
        if metadata_obj.shape == () or metadata_obj.size == 1:
            parsed = metadata_obj.item()
            return parsed if isinstance(parsed, dict) else {}
        return {}
    return metadata_obj if isinstance(metadata_obj, dict) else {}


def add_satellite_basemap(ax, crs_obj: CRS) -> None:
    try:
        ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, crs=crs_obj.to_string())
    except Exception as exc:
        print(f"Basemap fetch failed ({exc}). Continuing without basemap.")


PROJECT_ROOT = resolve_project_root()
SITES_YAML = PROJECT_ROOT / "configs/sites.yaml"
FULL_FIELD_NPZ = PROJECT_ROOT / "data/processed/bathy/bathy_field_full.npz"
SUBGRID_NPZ = PROJECT_ROOT / "data/processed/bathy/bathy_field_project_site.npz"

full_npz = np.load(FULL_FIELD_NPZ, allow_pickle=True)
sub_npz = np.load(SUBGRID_NPZ, allow_pickle=True)

full_x = full_npz["x"]
full_y = full_npz["y"]
full_depth = full_npz["z"]

x_vals = sub_npz["x"]
y_vals = sub_npz["y"]
depth_np = sub_npz["z"]

full_extent = [
    float(np.nanmin(full_x)),
    float(np.nanmax(full_x)),
    float(np.nanmin(full_y)),
    float(np.nanmax(full_y)),
]
extent = [
    float(np.nanmin(x_vals)),
    float(np.nanmax(x_vals)),
    float(np.nanmin(y_vals)),
    float(np.nanmax(y_vals)),
]

full_metadata = parse_npz_metadata(full_npz["metadata"])
sub_metadata = parse_npz_metadata(sub_npz["metadata"])

attr_epsg = parse_epsg(sub_metadata.get("epsg")) or parse_epsg(full_metadata.get("epsg"))
x_is_lon = float(np.nanmin(x_vals)) >= -180.0 and float(np.nanmax(x_vals)) <= 180.0
y_is_lat = float(np.nanmin(y_vals)) >= -90.0 and float(np.nanmax(y_vals)) <= 90.0

if attr_epsg is not None:
    data_crs = CRS.from_epsg(attr_epsg)
    crs_source = f"from NPZ metadata (EPSG:{attr_epsg})"
elif x_is_lon and y_is_lat:
    data_crs = CRS.from_epsg(4326)
    crs_source = "inferred from coordinate ranges as geographic WGS84 (EPSG:4326)"
else:
    data_crs = CRS.from_epsg(32633)
    crs_source = "fallback default UTM zone 33N (EPSG:32633)"

with SITES_YAML.open("r", encoding="utf-8") as handle:
    cfg = yaml.safe_load(handle)

site_records = []
for group in ("offshore_sites", "nearshore_sites"):
    for site in cfg.get(group, []):
        site_name = site["name"]
        is_nora_point = group == "offshore_sites" or site_name.lower().startswith("nora3")
        site_role = "nora_points" if is_nora_point else "project_sites"
        site_records.append(
            {
                "site_name": site_name,
                "site_group": group,
                "site_role": site_role,
                "lat": site["lat"],
                "lon": site["lon"],
            }
        )

sites_df = pd.DataFrame(site_records)

if data_crs.to_epsg() == 4326:
    sites_df["x"] = sites_df["lon"].to_numpy()
    sites_df["y"] = sites_df["lat"].to_numpy()
else:
    transformer = Transformer.from_crs(CRS.from_epsg(4326), data_crs, always_xy=True)
    sites_df["x"], sites_df["y"] = transformer.transform(
        sites_df["lon"].to_numpy(),
        sites_df["lat"].to_numpy(),
    )

sites_df["inside_subgrid"] = sites_df["x"].between(extent[0], extent[1]) & sites_df["y"].between(
    extent[2], extent[3]
)

valid_mask = np.isfinite(depth_np)
yy, xx = np.meshgrid(y_vals, x_vals, indexing="ij")
valid_xy = np.column_stack((xx[valid_mask], yy[valid_mask]))
valid_z = depth_np[valid_mask]

if valid_xy.size == 0:
    raise ValueError("No finite bathymetry cells available in the project subgrid.")

tree = cKDTree(valid_xy)
site_xy = sites_df[["x", "y"]].to_numpy()
distance_to_sample, nearest_idx = tree.query(site_xy, k=1)
sites_df["depth_m_from_grid"] = np.round(valid_z[nearest_idx], 2)
distance_col = "distance_to_sample_deg" if data_crs.is_geographic else "distance_to_sample_m"
sites_df[distance_col] = np.round(distance_to_sample, 3)

project_sites_df = sites_df[sites_df["site_role"] == "project_sites"]
if project_sites_df.empty:
    project_sites_df = sites_df.copy()

zoom_padding = 0.03 if data_crs.is_geographic else 5_000.0
zoom_extent = [
    max(extent[0], float(project_sites_df["x"].min()) - zoom_padding),
    min(extent[1], float(project_sites_df["x"].max()) + zoom_padding),
    max(extent[2], float(project_sites_df["y"].min()) - zoom_padding),
    min(extent[3], float(project_sites_df["y"].max()) + zoom_padding),
]

data_crs_label = (
    f"EPSG:{data_crs.to_epsg()}" if data_crs.to_epsg() is not None else data_crs.to_string()
)

print(f"Project root: {PROJECT_ROOT}")
print(f"Detected data CRS: {data_crs_label} ({crs_source})")
print(f"Full field shape (y, x): {full_depth.shape}")
print(f"Project subgrid shape (y, x): {depth_np.shape}")
print(f"Sites inside project subgrid: {int(sites_df['inside_subgrid'].sum())} / {len(sites_df)}")
print(f"NORA points: {int((sites_df['site_role'] == 'nora_points').sum())}")

sites_df.head()

In [ ]:
# Plot 1: Full field context with project sites and NORA points
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(full_extent[0], full_extent[1])
ax.set_ylim(full_extent[2], full_extent[3])
add_satellite_basemap(ax, data_crs)

# Downsample for faster rendering while preserving geographic context.
stride = max(1, int(max(full_depth.shape) // 1_500))
img = ax.imshow(
    full_depth[::stride, ::stride],
    extent=full_extent,
    origin="lower",
    cmap="viridis_r",
    alpha=0.55,
    zorder=2,
)

project_df = sites_df[sites_df["site_role"] == "project_sites"]
nora_df = sites_df[sites_df["site_role"] == "nora_points"]

if not project_df.empty:
    ax.scatter(
        project_df["x"],
        project_df["y"],
        s=45,
        c="#3ddad7",
        edgecolor="black",
        linewidth=0.7,
        label="Project site points",
        zorder=3,
    )

if not nora_df.empty:
    ax.scatter(
        nora_df["x"],
        nora_df["y"],
        s=80,
        marker="*",
        c="#ffb703",
        edgecolor="black",
        linewidth=0.8,
        label="NORA points",
        zorder=4,
    )

cbar = fig.colorbar(img, ax=ax, fraction=0.036, pad=0.02)
cbar.set_label("Bathymetry depth/elevation (m)")
ax.legend(loc="upper right")
ax.set_title("Full Bathymetry Field with NORA Points")
axis_unit = "degrees" if data_crs.is_geographic else "meters"
ax.set_xlabel(f"X ({axis_unit}), {data_crs_label}")
ax.set_ylabel(f"Y ({axis_unit}), {data_crs_label}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2: Project subgrid with explicit NORA point overlay
fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(zoom_extent[0], zoom_extent[1])
ax.set_ylim(zoom_extent[2], zoom_extent[3])
add_satellite_basemap(ax, data_crs)

img = ax.imshow(
    depth_np,
    extent=extent,
    origin="lower",
    cmap="viridis_r",
    alpha=0.62,
    zorder=2,
)

sites_in_subgrid = sites_df[sites_df["inside_subgrid"]].copy()
project_in = sites_in_subgrid[sites_in_subgrid["site_role"] == "project_sites"]
nora_in = sites_in_subgrid[sites_in_subgrid["site_role"] == "nora_points"]

if not project_in.empty:
    ax.scatter(
        project_in["x"],
        project_in["y"],
        s=55,
        c="#3ddad7",
        edgecolor="black",
        linewidth=0.8,
        label="Project site points",
        zorder=3,
    )

if not nora_in.empty:
    ax.scatter(
        nora_in["x"],
        nora_in["y"],
        s=95,
        marker="*",
        c="#fb8500",
        edgecolor="black",
        linewidth=0.9,
        label="NORA points",
        zorder=4,
    )

cbar = fig.colorbar(img, ax=ax, fraction=0.036, pad=0.02)
cbar.set_label("Bathymetry depth/elevation (m)")
ax.legend(loc="upper right")
ax.set_title("Project Site Bathymetry + NORA Points")
axis_unit = "degrees" if data_crs.is_geographic else "meters"
ax.set_xlabel(f"X ({axis_unit}), {data_crs_label}")
ax.set_ylabel(f"Y ({axis_unit}), {data_crs_label}")
plt.tight_layout()
plt.show()

In [ ]:
# Plot 3: Land-mask style QA overlay on the project subgrid
finite = depth_np[np.isfinite(depth_np)]

if finite.size == 0:
    raise ValueError("No finite bathymetry values found in project subgrid.")

# Infer sign convention: if almost all values are positive, assume positive-down depth.
if np.mean(finite > 0) > 0.9:
    land_mask = np.isnan(depth_np) | (depth_np <= 0.0)
    mask_note = "Assumed positive-down depth (land <= 0 or NaN)"
else:
    land_mask = np.isnan(depth_np) | (depth_np >= 0.0)
    mask_note = "Assumed elevation convention (land >= 0 or NaN)"

land_rgba = np.zeros((*land_mask.shape, 4), dtype=float)
land_rgba[..., 0] = 1.0
land_rgba[..., 1] = 0.12
land_rgba[..., 2] = 0.12
land_rgba[..., 3] = land_mask.astype(float) * 0.9

fig, ax = plt.subplots(figsize=(12, 10))
ax.set_xlim(extent[0], extent[1])
ax.set_ylim(extent[2], extent[3])
add_satellite_basemap(ax, data_crs)

ax.imshow(land_rgba, extent=extent, origin="lower", zorder=2)

project_in = sites_df[(sites_df["site_role"] == "project_sites") & (sites_df["inside_subgrid"])]
nora_in = sites_df[(sites_df["site_role"] == "nora_points") & (sites_df["inside_subgrid"])]

if not project_in.empty:
    ax.scatter(
        project_in["x"],
        project_in["y"],
        s=30,
        c="#3ddad7",
        edgecolor="black",
        linewidth=0.5,
        label="Project site points",
        zorder=3,
    )

if not nora_in.empty:
    ax.scatter(
        nora_in["x"],
        nora_in["y"],
        s=75,
        marker="*",
        c="#ffb703",
        edgecolor="black",
        linewidth=0.7,
        label="NORA points",
        zorder=4,
    )

ax.set_title("Project Subgrid Land Mask QA")
axis_unit = "degrees" if data_crs.is_geographic else "meters"
ax.set_xlabel(f"X ({axis_unit}), {data_crs_label}")
ax.set_ylabel(f"Y ({axis_unit}), {data_crs_label}")
ax.text(
    0.01,
    0.01,
    mask_note,
    transform=ax.transAxes,
    color="white",
    fontsize=10,
    bbox={"facecolor": "black", "alpha": 0.6, "pad": 4},
)
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
# Table: bathymetry sampled at configured points (including NORA points)
distance_col = "distance_to_sample_deg" if data_crs.is_geographic else "distance_to_sample_m"
bathymetry_table = (
    sites_df[
        [
            "site_name",
            "site_group",
            "site_role",
            "lat",
            "lon",
            "x",
            "y",
            "inside_subgrid",
            "depth_m_from_grid",
            distance_col,
        ]
    ]
    .sort_values(["site_role", "site_name"])
    .reset_index(drop=True)
)

print("Bathymetry sampled at point locations (nearest finite project-subgrid cell):")
bathymetry_table